# Gemma 4 E2B Text SFT — Official Unsloth Pattern (2026)

This notebook is a direct port of the official `Gemma4_(E2B)_Text.ipynb`. Designed to be read side-by-side with `01_sft_modern.ipynb` (Qwen3).

## Qwen3 vs Gemma 4 — Practical Differences

| | Qwen3-4B-Instruct | Gemma 4 E2B-it |
|---|---|---|
| **Class** | `FastLanguageModel` | **`FastModel`** (multimodal-native) |
| **PEFT API** | `target_modules=[...]` list | **`finetune_*` flags** (vision/lang/attn/mlp) |
| **LoRA** | r=32, alpha=32 | **r=8, alpha=8** (small model) |
| **Chat template** | `qwen3-instruct` | **`gemma-4`** |
| **Markers** | `<|im_start|>user\n` / `<|im_start|>assistant\n` | **`<|turn>user\n` / `<|turn>model\n`** |
| **Format quirk** | none | **`removeprefix('<bos>')`** mandatory |
| **Generation** | T=0.7, top_p=0.8, top_k=20 | **T=1.0, top_p=0.95, top_k=64** |

**Hardware:** 16GB GPU is enough (Gemma 4 E2B = ~2B activated params)

## 1. Setup

In [ ]:
import unsloth                                # MUST be the very first import
from unsloth import FastModel                  # NOT FastLanguageModel — Gemma 4 is multimodal-native
from unsloth.chat_templates import (
    get_chat_template,
    standardize_data_formats,
    train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 2. Model — Gemma 4 E2B Instruct

Options from the official list:

**Instruct (chat-tuned):**
- `unsloth/gemma-4-E2B-it` — ~2B activated, fits easily on a 16GB GPU
- `unsloth/gemma-4-E4B-it` — ~4B activated
- `unsloth/gemma-4-31B-it` — large
- `unsloth/gemma-4-26B-A4B-it` — MoE

**Base (pre-trained, no instruction-following):**
- `unsloth/gemma-4-E2B`
- `unsloth/gemma-4-E4B`

**Important:** `dtype = None` — Unsloth picks bf16/fp16 automatically. `load_in_4bit = False` — Gemma 4 E2B fits in 16GB even with 16-bit LoRA.

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None,                           # auto-detect (bf16 if Ampere+)
    max_seq_length = 1024,                   # raise for longer context
    load_in_4bit = False,                   # 16-bit LoRA (fits in 16GB). True = 4-bit QLoRA
    full_finetuning = False,                # [NEW!] full FT now available
    # token = "YOUR_HF_TOKEN",              # for gated models
)

## 3. LoRA — `finetune_*` Flags

Because Gemma 4 is multimodal-native, `FastModel.get_peft_model` differs from Qwen3:

- **No** `target_modules` list
- Instead, four `finetune_*` boolean flags
- For text-only SFT use `finetune_vision_layers = False`
- Small r and alpha (8) — model is E2B (~2B activated)

**RSLoRA / DoRA / LoftQ** can be enabled via `use_rslora = True` or `loftq_config = ...` (the official notebook doesn't use these).

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,    # Text-only SFT — freeze the vision tower
    finetune_language_layers   = True,     # LM body — on
    finetune_attention_modules = True,     # Attention — also good for GRPO
    finetune_mlp_modules       = True,     # MLP — always on

    r = 8,                                  # Larger = higher accuracy but overfit risk
    lora_alpha = 8,                         # alpha == r recommended
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 4. Chat Template — `gemma-4`

Gemma 4's markers are **different from Qwen3**:

```
<bos><|turn>user
Question<turn|>
<|turn>model
Answer<turn|>
```

Watch out for the asymmetric markers: opening is `<|turn>` but closing is `<turn|>`.

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

# Verify
sample_msgs = [
    {"role": "user", "content": "What is interest?"},
    {"role": "assistant", "content": "Interest is the time value of money."},
]
formatted = tokenizer.apply_chat_template(sample_msgs, tokenize=False)
print(formatted)

## 5. Data — FineTome-100k (the dataset used by the official notebook)

We're using Maxime Labonne's FineTome-100k dataset (ShareGPT format). First 3000 rows.

**Flow:** raw → `standardize_data_formats` (ShareGPT/Alpaca → conversations) → `formatting_prompts_func` (apply chat template + strip `<bos>`)

In [ ]:
dataset = load_dataset("mlabonne/FineTome-100k", split = "train[:3000]")
print(f'Raw rows: {len(dataset)}')

# standardize_data_formats: ShareGPT/Alpaca → 'conversations' column
dataset = standardize_data_formats(dataset)
print(dataset[0])

### `formatting_prompts_func` — `<bos>` Strip Is MANDATORY

The chat template adds `<bos>`, but the processor will also add `<bos>` before training. Double `<bos>` breaks the model — we clean it up with `removeprefix('<bos>')`.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
        for c in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:400])

## 6. SFTTrainer

Settings identical to the official notebook:
- `weight_decay = 0.001` (official default — not 0.01)
- `lr_scheduler_type = "linear"` (not cosine)
- `warmup_steps = 5`
- `optim = "adamw_8bit"`
- `seed = 3407`

**Smoke test:** `max_steps = 60` (production: `num_train_epochs = 1`)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,       # effective batch = 4
        warmup_steps = 5,
        max_steps = 60,                         # demo; production: num_train_epochs=1
        learning_rate = 2e-4,                   # for LoRA. Long training: 2e-5
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,                   # 0.001 — official default
        lr_scheduler_type = "linear",           # linear — official default
        seed = 3407,
        report_to = "none",
    ),
)

## 7. `train_on_responses_only` — Gemma 4 Markers

**IMPORTANT:** Gemma 4's markers differ from Qwen3:
- Qwen3: `<|im_start|>user\n` / `<|im_start|>assistant\n`
- **Gemma 4: `<|turn>user\n` / `<|turn>model\n`** (assistant role is `model`)

If you use the wrong marker, masking silently does nothing — loss is computed across the full sequence (the user input gets learned too, which is bad). Always verify with decode.

In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",     # Gemma 4 marker
    response_part    = "<|turn>model\n",    # Gemma 4 marker (not assistant — 'model'!)
)

# Masking verification — sample 100
sample_idx = min(100, len(trainer.train_dataset) - 1)
print("=== FULL INPUT (instruction + response) ===")
print(tokenizer.decode(trainer.train_dataset[sample_idx]["input_ids"]))

print("\n=== ONLY UNMASKED LABELS (should only show the response) ===")
print(tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in trainer.train_dataset[sample_idx]["labels"]
]).replace(tokenizer.pad_token, " "))

## 8. Training

In [ ]:
# Memory snapshot — as in the official notebooks
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"\n{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")

## 9. Inference

Recommended Gemma 4 generation params: `temperature=1.0, top_p=0.95, top_k=64`.

Because Gemma 4 is multimodal, the input format differs from Qwen3 — `content` must be a list in the form `{"type": "text", "text": ...}` (even for text-only use).

In [ ]:
from transformers import TextStreamer

messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "Continue the sequence: 1, 1, 2, 3, 5, 8,"}],
}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,           # MANDATORY for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens = 128,
    temperature = 1.0, top_p = 0.95, top_k = 64,    # Gemma 4 recommendation
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

## 10. Save & Deploy

In [ ]:
# A. LoRA adapter (smallest)
model.save_pretrained("gemma4_e2b_lora")
tokenizer.save_pretrained("gemma4_e2b_lora")

# B. Merged 16-bit (vLLM/HF inference)
# model.save_pretrained_merged(
#     "gemma4_e2b_merged",
#     tokenizer,
#     save_method = "merged_16bit",
# )

# C. GGUF (Ollama/llama.cpp)
# model.save_pretrained_gguf(
#     "gemma4_e2b_gguf",
#     tokenizer,
#     quantization_method = "q4_k_m",
# )

# D. Push to Hub
# model.push_to_hub("USER/gemma4_e2b_lora", token="hf_xxx")

print("LoRA saved: gemma4_e2b_lora/")

## Important Notes

1. **`<bos>` strip** — `removeprefix('<bos>')` inside `formatting_prompts_func` is mandatory. Without it, double bos = broken model.
2. **Markers differ from Qwen3** — `<|turn>user\n` / `<|turn>model\n`. The assistant role isn't `assistant` but **`model`**.
3. **No `tools=` support** — Gemma 4's chat template has no `tools` branch. If you need tool calling, embed the JSON schema manually in the system prompt.
4. **`enable_thinking`** — `gemma-4-thinking` is a separate template (this notebook uses standard Gemma 4).
5. **`finetune_vision_layers = False`** — mandatory for text-only, otherwise the vision encoder also gets updated (wasted VRAM + harm).
6. **`content` format at generation time** — must be a `[{"type": "text", "text": ...}]` list. Passing a plain string can throw an error.